In [1]:
%pip install sqlalchemy pymysql

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   - -------------------------------------- 0.1/2.4 MB 383.3 kB/s eta 0:00:07
   - -------------------------------------- 0.1/2.4 MB 383.3 kB/s eta 0:00:07
   -- ------------------------------------- 0.2/2.4 MB 459.5 kB/s eta 0:00:05
   --- ------------------------------------ 0.2/2.4 MB 506.0 kB/s eta 0:00:05
   ---- ----------------------------------- 0.3/2.4 MB 708.9 kB/s eta 0:00:04
   ------ --------------------------------- 0.4/2.4 MB 836.4 kB/s eta 0:00:03
   ------- -------------------------------- 0.5/2.4 MB 982.5 kB/s eta 0:00:03
   ------- -------------------------------- 0.5/2.4 MB 982.5 kB/s eta 0:00:03
   ------- --------------------


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from getpass import getpass

username = input("MySQL username: ")
password = getpass("MySQL password: ")

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=username,
    password=password,
    host="localhost",
    port=3306,
    database="freshcart_analytics"
)

engine = create_engine(connection_url)

with engine.connect() as connection:
    result = connection.execute(
        text("SELECT DATABASE(), VERSION()")
    )
    print("Connected successfully!")
    print("Database and MySQL version:", result.fetchone())

Connected successfully!
Database and MySQL version: ('freshcart_analytics', '8.0.44')


In [3]:
from sqlalchemy import text

create_staging_tables = [
    """
    CREATE TABLE IF NOT EXISTS stg_inventory (
        Product_No VARCHAR(50),
        Store VARCHAR(50),
        Supplier VARCHAR(100),
        Start_Date DATE,
        End_Date DATE,
        Stock_Status VARCHAR(50),
        Qty_on_hand INT,
        Stocks_Selling_Amount DECIMAL(14,2),
        Cost_of_Stocks DECIMAL(14,2),
        Stock_Unit_Selling_Price DECIMAL(12,2),
        Stock_Unit_Cost_Price DECIMAL(12,2),
        Product_Description TEXT,
        Product_Division VARCHAR(150),
        Product_Category VARCHAR(150),
        Product_Subcategory VARCHAR(150),
        Product_Segment VARCHAR(200),
        Store_Type VARCHAR(100),
        Sales_Channel VARCHAR(100),
        flag_negative_stock BOOLEAN,
        flag_negative_unit_cost BOOLEAN,
        flag_zero_selling_price BOOLEAN,
        flag_zero_qty_residual_cost BOOLEAN,
        calculated_selling_value DECIMAL(14,2),
        calculated_cost_value DECIMAL(14,2)
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS stg_demand (
        Date DATE,
        Store_ID VARCHAR(20),
        Product_ID VARCHAR(20),
        Category VARCHAR(100),
        Region VARCHAR(50),
        Inventory_Level INT,
        Units_Sold INT,
        Units_Ordered INT,
        Demand_Forecast DECIMAL(12,2),
        Price DECIMAL(12,2),
        Discount DECIMAL(8,2),
        Weather_Condition VARCHAR(50),
        Holiday_Promotion BOOLEAN,
        Competitor_Pricing DECIMAL(12,2),
        Seasonality VARCHAR(50),
        flag_negative_forecast BOOLEAN,
        Demand_Forecast_Clean DECIMAL(12,2),
        Inventory_Coverage DECIMAL(12,2)
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS stg_sales (
        Transaction_Date DATE,
        Sales_Type VARCHAR(50),
        Is_Return BOOLEAN,
        Reason_of_Return VARCHAR(200),
        Supplier VARCHAR(100),
        Product_No VARCHAR(50),
        Product_Description TEXT,
        Product_Division VARCHAR(150),
        Product_Category VARCHAR(150),
        Product_Subcategory VARCHAR(150),
        Product_Segment VARCHAR(200),
        Store VARCHAR(50),
        Sales_Channel VARCHAR(100),
        Qty_Sold DECIMAL(12,2),
        Sales_Amount DECIMAL(14,2),
        Cogs DECIMAL(14,2),
        Number_of_Transactions INT,
        flag_negative_quantity BOOLEAN,
        flag_negative_sales BOOLEAN,
        flag_negative_cogs BOOLEAN,
        Net_Profit_Contribution DECIMAL(14,2),
        Net_Quantity DECIMAL(12,2),
        Net_Sales DECIMAL(14,2),
        Net_COGS DECIMAL(14,2)
    )
    """
]

with engine.begin() as connection:
    for statement in create_staging_tables:
        connection.execute(text(statement))

print("SUCCESS: All three staging tables created!")

SUCCESS: All three staging tables created!


In [5]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

inventory_df = pd.read_csv(
    PROCESSED_DIR / "inventory_clean.csv"
)

# Match database column names
inventory_df.columns = [
    col.replace(" ", "_").replace("/", "_")
    for col in inventory_df.columns
]

print("Rows ready:", len(inventory_df))
print("Columns ready:", len(inventory_df.columns))

inventory_df.to_sql(
    name="stg_inventory",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000,
    method=None
)

print("Inventory loaded successfully!")

Rows ready: 284755
Columns ready: 24
Inventory loaded successfully!


In [7]:
with engine.connect() as connection:
    count = connection.execute(
        text("SELECT COUNT(*) FROM stg_inventory")
    ).scalar()

    print("Inventory records in MySQL:", count)

Inventory records in MySQL: 284755


In [9]:
demand_df = pd.read_csv(
    PROCESSED_DIR / "demand_clean.csv"
)

demand_df.columns = [
    col.replace(" ", "_").replace("/", "_")
    for col in demand_df.columns
]

print("Demand rows ready:", len(demand_df))

demand_df.to_sql(
    name="stg_demand",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000,
    method=None
)

print("Demand dataset loaded successfully!")

Demand rows ready: 73100
Demand dataset loaded successfully!


In [10]:
with engine.connect() as connection:
    count = connection.execute(
        text("SELECT COUNT(*) FROM stg_demand")
    ).scalar()

    print("Demand records:", count)

Demand records: 73100


In [12]:
sales_df = pd.read_csv(
    PROCESSED_DIR / "sales_clean.csv"
)

sales_df.columns = [
    col.replace(" ", "_").replace("/", "_")
    for col in sales_df.columns
]

print("Sales rows ready:", len(sales_df))

sales_df.to_sql(
    name="stg_sales",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000,
    method=None
)

print("Sales dataset loaded successfully!")

Sales rows ready: 125751
Sales dataset loaded successfully!


In [13]:
with engine.connect() as connection:
    count = connection.execute(
        text("SELECT COUNT(*) FROM stg_sales")
    ).scalar()

    print("Sales records in MySQL:", count)

Sales records in MySQL: 125751


In [14]:
from sqlalchemy import text

with engine.connect() as connection:
    for table in ["stg_inventory", "stg_demand", "stg_sales"]:
        count = connection.execute(
            text(f"SELECT COUNT(*) FROM {table}")
        ).scalar()

        print(f"{table}: {count:,} rows")

stg_inventory: 284,755 rows
stg_demand: 73,100 rows
stg_sales: 125,751 rows


In [15]:
from sqlalchemy import text
import pandas as pd

expected_counts = {
    "stg_inventory": 284755,
    "stg_demand": 73100,
    "stg_sales": 125751
}

audit_results = []

with engine.connect() as connection:

    for table, expected in expected_counts.items():

        actual = connection.execute(
            text(f"SELECT COUNT(*) FROM {table}")
        ).scalar()

        audit_results.append({
            "Table": table,
            "Expected Rows": expected,
            "Actual Rows": actual,
            "Difference": actual - expected,
            "Status": "PASS" if actual == expected else "FAIL"
        })

audit_df = pd.DataFrame(audit_results)

display(audit_df)

print("\nTOTAL RECORDS IN MYSQL:",
      audit_df["Actual Rows"].sum())

print("\nDATABASE AUDIT:",
      "PASSED" if (audit_df["Status"] == "PASS").all()
      else "FAILED")

,Table,Expected Rows,Actual Rows,Difference,Status
0,stg_inventory,284755,284755,0,PASS
1,stg_demand,73100,73100,0,PASS
2,stg_sales,125751,125751,0,PASS



TOTAL RECORDS IN MYSQL: 483606

DATABASE AUDIT: PASSED


In [16]:
from sqlalchemy import text

create_schema = """
CREATE SCHEMA IF NOT EXISTS freshcart_analytics;
"""

with engine.begin() as connection:
    connection.execute(text(create_schema))

print("FreshCart analytics schema verified!")

FreshCart analytics schema verified!


In [17]:
with engine.connect() as connection:

    tables = connection.execute(text("""
        SHOW TABLES
    """)).fetchall()

    print("TABLES IN FRESHCART_ANALYTICS:")

    for table in tables:
        print(table[0])

TABLES IN FRESHCART_ANALYTICS:
stg_demand
stg_inventory
stg_sales


In [18]:
with engine.begin() as connection:
    connection.execute(text("""
        CREATE TABLE IF NOT EXISTS dim_product (
            product_key INT AUTO_INCREMENT PRIMARY KEY,
            product_no VARCHAR(50) NOT NULL UNIQUE,
            product_description TEXT,
            product_division VARCHAR(150),
            product_category VARCHAR(150),
            product_subcategory VARCHAR(150),
            product_segment VARCHAR(200)
        )
    """))

print("dim_product created successfully!")

dim_product created successfully!


In [19]:
with engine.begin() as connection:
    connection.execute(text("""
        INSERT INTO dim_product (
            product_no,
            product_description,
            product_division,
            product_category,
            product_subcategory,
            product_segment
        )
        SELECT
            Product_No,
            MAX(Product_Description),
            MAX(Product_Division),
            MAX(Product_Category),
            MAX(Product_Subcategory),
            MAX(Product_Segment)
        FROM stg_inventory
        WHERE Product_No IS NOT NULL
        GROUP BY Product_No
    """))

print("dim_product populated successfully!")

dim_product populated successfully!


In [20]:
with engine.connect() as connection:
    count = connection.execute(
        text("SELECT COUNT(*) FROM dim_product")
    ).scalar()

    print("Products in dimension:", count)

    sample = pd.read_sql(
        "SELECT * FROM dim_product LIMIT 10",
        connection
    )

display(sample)

Products in dimension: 2326


,product_key,product_no,product_description,product_division,product_category,product_subcategory,product_segment
0,1,PROD-100043,PROD-189472 SEG-520541 Femme Footwear Boot Col...,Femme Footwear,Femme Footwear Boot Collection,Femme Footwear Boot Collection Plushfoot,SEG-520541 Femme Footwear Boot Collection Plus...
1,2,PROD-100044,PROD-189473 SEG-520541 Femme Footwear Boot Col...,Femme Footwear,Femme Footwear Boot Collection,Femme Footwear Boot Collection Plushfoot,SEG-520541 Femme Footwear Boot Collection Plus...
2,3,PROD-100045,PROD-189474 SEG-520541 Femme Footwear Boot Col...,Femme Footwear,Femme Footwear Boot Collection,Femme Footwear Boot Collection Plushfoot,SEG-520541 Femme Footwear Boot Collection Plus...
3,4,PROD-100051,PROD-189481 SEG-520542 Femme Footwear Boot Col...,Femme Footwear,Femme Footwear Boot Collection,Femme Footwear Boot Collection Plushfoot,SEG-520542 Femme Footwear Boot Collection Plus...
4,5,PROD-100052,PROD-189475 SEG-520542 Femme Footwear Boot Col...,Femme Footwear,Femme Footwear Boot Collection,Femme Footwear Boot Collection Plushfoot,SEG-520542 Femme Footwear Boot Collection Plus...
5,6,PROD-100353,PROD-164489 SEG-515310 Femme Footwear Court Ca...,Femme Footwear,Femme Footwear Court Casual,Femme Footwear Court Casual Zephyr,SEG-515310 Femme Footwear Court Casual Zephyr
6,7,PROD-100354,PROD-164490 SEG-515310 Femme Footwear Court Ca...,Femme Footwear,Femme Footwear Court Casual,Femme Footwear Court Casual Zephyr,SEG-515310 Femme Footwear Court Casual Zephyr
7,8,PROD-100355,PROD-164491 SEG-515310 Femme Footwear Court Ca...,Femme Footwear,Femme Footwear Court Casual,Femme Footwear Court Casual Zephyr,SEG-515310 Femme Footwear Court Casual Zephyr
8,9,PROD-100356,PROD-164492 SEG-515310 Femme Footwear Court Ca...,Femme Footwear,Femme Footwear Court Casual,Femme Footwear Court Casual Zephyr,SEG-515310 Femme Footwear Court Casual Zephyr
9,10,PROD-100357,PROD-164493 SEG-515310 Femme Footwear Court Ca...,Femme Footwear,Femme Footwear Court Casual,Femme Footwear Court Casual Zephyr,SEG-515310 Femme Footwear Court Casual Zephyr


In [21]:
with engine.begin() as connection:
    connection.execute(text("""
        CREATE TABLE IF NOT EXISTS dim_store (
            store_key INT AUTO_INCREMENT PRIMARY KEY,
            store_id VARCHAR(50) NOT NULL UNIQUE,
            store_type VARCHAR(100),
            sales_channel VARCHAR(100)
        )
    """))

print("dim_store created!")

dim_store created!


In [22]:
with engine.begin() as connection:
    connection.execute(text("""
        INSERT INTO dim_store (
            store_id,
            store_type,
            sales_channel
        )
        SELECT
            Store,
            MAX(Store_Type),
            MAX(Sales_Channel)
        FROM stg_inventory
        WHERE Store IS NOT NULL
        GROUP BY Store
    """))

print("dim_store populated!")

dim_store populated!


In [25]:
with engine.connect() as connection:
    store_count = connection.execute(
        text("SELECT COUNT(*) FROM dim_store")
    ).scalar()

    print("Unique stores:", store_count)

    display(pd.read_sql(
        "SELECT * FROM dim_store ORDER BY store_id LIMIT 10",
        connection
    ))

Unique stores: 40


,store_key,store_id,store_type,sales_channel
0,30,STR-1006,Retail Storefront,Channel Alpha
1,1,STR-1010,Retail Storefront,Channel Alpha
2,34,STR-1026,Retail Storefront,Channel Alpha
3,40,STR-1038,Retail Storefront,Channel Alpha
4,17,STR-1044,Retail Storefront,Channel Alpha
5,7,STR-1074,Retail Storefront,Channel Alpha
6,12,STR-1075,Retail Storefront,Channel Alpha
7,28,STR-1080,Retail Storefront,Channel Alpha
8,2,STR-1082,Retail Storefront,Channel Alpha
9,29,STR-1085,Retail Storefront,Channel Alpha


In [24]:
with engine.begin() as connection:
    connection.execute(text("""
        CREATE TABLE IF NOT EXISTS dim_supplier (
            supplier_key INT AUTO_INCREMENT PRIMARY KEY,
            supplier_id VARCHAR(100) NOT NULL UNIQUE,
            supplier_name VARCHAR(150)
        )
    """))

print("dim_supplier created successfully!")

dim_supplier created successfully!


In [26]:
with engine.begin() as connection:
    connection.execute(text("""
        INSERT INTO dim_supplier (
            supplier_id,
            supplier_name
        )
        SELECT DISTINCT
            Supplier,
            Supplier
        FROM stg_inventory
        WHERE Supplier IS NOT NULL
    """))

print("dim_supplier populated successfully!")

dim_supplier populated successfully!


In [27]:
with engine.connect() as connection:
    supplier_count = connection.execute(
        text("SELECT COUNT(*) FROM dim_supplier")
    ).scalar()

    print("Unique suppliers:", supplier_count)

    display(pd.read_sql(
        "SELECT * FROM dim_supplier ORDER BY supplier_id",
        connection
    ))

Unique suppliers: 30


,supplier_key,supplier_id,supplier_name
0,13,Vendor 0008,Vendor 0008
1,2,Vendor 0010,Vendor 0010
2,24,Vendor 0025,Vendor 0025
3,29,Vendor 0027,Vendor 0027
4,8,Vendor 0037,Vendor 0037
5,27,Vendor 0039,Vendor 0039
6,30,Vendor 0042,Vendor 0042
7,23,Vendor 0052,Vendor 0052
8,6,Vendor 0054,Vendor 0054
9,11,Vendor 0064,Vendor 0064


In [28]:
with engine.begin() as connection:
    connection.execute(text("""
        CREATE TABLE IF NOT EXISTS dim_date (
            date_key INT PRIMARY KEY,
            full_date DATE NOT NULL UNIQUE,
            year INT,
            quarter INT,
            month INT,
            month_name VARCHAR(20),
            week_number INT,
            day_of_month INT,
            day_name VARCHAR(20),
            day_of_week INT,
            is_weekend BOOLEAN
        )
    """))

print("dim_date created successfully!")

dim_date created successfully!


In [29]:
with engine.begin() as connection:
    connection.execute(text("""
        INSERT INTO dim_date (
            date_key, full_date, year, quarter,
            month, month_name, week_number,
            day_of_month, day_name, day_of_week,
            is_weekend
        )
        SELECT
            CAST(DATE_FORMAT(d, '%Y%m%d') AS UNSIGNED),
            d,
            YEAR(d),
            QUARTER(d),
            MONTH(d),
            MONTHNAME(d),
            WEEK(d, 3),
            DAY(d),
            DAYNAME(d),
            DAYOFWEEK(d),
            CASE WHEN DAYOFWEEK(d) IN (1, 7)
                 THEN TRUE ELSE FALSE END
        FROM (
            SELECT DISTINCT calendar_date AS d
            FROM (
                SELECT Date AS calendar_date FROM stg_demand
                UNION
                SELECT Transaction_Date FROM stg_sales
                UNION
                SELECT Start_Date FROM stg_inventory
                UNION
                SELECT End_Date FROM stg_inventory
            ) all_dates
            WHERE calendar_date IS NOT NULL
        ) unique_dates
    """))

print("dim_date populated successfully!")

dim_date populated successfully!


In [30]:
with engine.connect() as connection:
    date_count = connection.execute(
        text("SELECT COUNT(*) FROM dim_date")
    ).scalar()

    print("Dates loaded:", date_count)

    display(pd.read_sql("""
        SELECT *
        FROM dim_date
        ORDER BY full_date
        LIMIT 10
    """, connection))

Dates loaded: 1060


,date_key,full_date,year,quarter,month,month_name,week_number,day_of_month,day_name,day_of_week,is_weekend
0,20220101,2022-01-01,2022,1,1,January,52,1,Saturday,7,1
1,20220102,2022-01-02,2022,1,1,January,52,2,Sunday,1,1
2,20220103,2022-01-03,2022,1,1,January,1,3,Monday,2,0
3,20220104,2022-01-04,2022,1,1,January,1,4,Tuesday,3,0
4,20220105,2022-01-05,2022,1,1,January,1,5,Wednesday,4,0
5,20220106,2022-01-06,2022,1,1,January,1,6,Thursday,5,0
6,20220107,2022-01-07,2022,1,1,January,1,7,Friday,6,0
7,20220108,2022-01-08,2022,1,1,January,1,8,Saturday,7,1
8,20220109,2022-01-09,2022,1,1,January,1,9,Sunday,1,1
9,20220110,2022-01-10,2022,1,1,January,2,10,Monday,2,0


In [31]:
with engine.begin() as connection:
    connection.execute(text("""
        CREATE TABLE IF NOT EXISTS fact_inventory (
            inventory_key BIGINT AUTO_INCREMENT PRIMARY KEY,
            product_key INT NOT NULL,
            store_key INT NOT NULL,
            supplier_key INT NOT NULL,
            start_date_key INT,
            end_date_key INT,
            stock_status VARCHAR(50),
            quantity_on_hand INT,
            selling_value DECIMAL(14,2),
            cost_value DECIMAL(14,2),
            unit_selling_price DECIMAL(12,2),
            unit_cost_price DECIMAL(12,2),

            FOREIGN KEY (product_key)
                REFERENCES dim_product(product_key),

            FOREIGN KEY (store_key)
                REFERENCES dim_store(store_key),

            FOREIGN KEY (supplier_key)
                REFERENCES dim_supplier(supplier_key),

            FOREIGN KEY (start_date_key)
                REFERENCES dim_date(date_key),

            FOREIGN KEY (end_date_key)
                REFERENCES dim_date(date_key)
        )
    """))

print("fact_inventory created successfully!")

fact_inventory created successfully!


In [32]:
with engine.begin() as connection:
    connection.execute(text("""
        INSERT INTO fact_inventory (
            product_key,
            store_key,
            supplier_key,
            start_date_key,
            end_date_key,
            stock_status,
            quantity_on_hand,
            selling_value,
            cost_value,
            unit_selling_price,
            unit_cost_price
        )
        SELECT
            p.product_key,
            st.store_key,
            su.supplier_key,
            CAST(DATE_FORMAT(i.Start_Date, '%Y%m%d') AS UNSIGNED),
            CAST(DATE_FORMAT(i.End_Date, '%Y%m%d') AS UNSIGNED),
            i.Stock_Status,
            i.Qty_on_hand,
            i.Stocks_Selling_Amount,
            i.Cost_of_Stocks,
            i.Stock_Unit_Selling_Price,
            i.Stock_Unit_Cost_Price
        FROM stg_inventory i
        INNER JOIN dim_product p
            ON i.Product_No = p.product_no
        INNER JOIN dim_store st
            ON i.Store = st.store_id
        INNER JOIN dim_supplier su
            ON i.Supplier = su.supplier_id
        LEFT JOIN dim_date ds
            ON i.Start_Date = ds.full_date
        LEFT JOIN dim_date de
            ON i.End_Date = de.full_date
    """))

print("fact_inventory populated successfully!")

fact_inventory populated successfully!


In [33]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            COUNT(*) AS total_records,
            SUM(quantity_on_hand) AS total_stock,
            ROUND(SUM(selling_value), 2) AS total_selling_value,
            ROUND(SUM(cost_value), 2) AS total_inventory_cost
        FROM fact_inventory
    """))

    print(result.fetchone())

(284755, Decimal('653407'), Decimal('63896905.11'), Decimal('31620976.61'))


In [34]:
from sqlalchemy import text
import pandas as pd

audit_query = """
SELECT
    (SELECT COUNT(*) FROM dim_product) AS products,
    (SELECT COUNT(*) FROM dim_store) AS stores,
    (SELECT COUNT(*) FROM dim_supplier) AS suppliers,
    (SELECT COUNT(*) FROM dim_date) AS dates,
    (SELECT COUNT(*) FROM fact_inventory) AS inventory_records
"""

with engine.connect() as connection:
    display(pd.read_sql(audit_query, connection))

,products,stores,suppliers,dates,inventory_records
0,2326,40,30,1060,284755


In [35]:
audit_query = """
SELECT 'dim_product' AS table_name, COUNT(*) AS total
FROM dim_product

UNION ALL

SELECT 'dim_store', COUNT(*) FROM dim_store

UNION ALL

SELECT 'dim_supplier', COUNT(*) FROM dim_supplier

UNION ALL

SELECT 'dim_date', COUNT(*) FROM dim_date

UNION ALL

SELECT 'fact_inventory', COUNT(*) FROM fact_inventory

UNION ALL

SELECT 'stg_inventory', COUNT(*) FROM stg_inventory
"""

with engine.connect() as connection:
    display(pd.read_sql(audit_query, connection))

,table_name,total
0,dim_product,2326
1,dim_store,40
2,dim_supplier,30
3,dim_date,1060
4,fact_inventory,284755
5,stg_inventory,284755


In [36]:
with engine.connect() as connection:
    display(pd.read_sql("DESCRIBE stg_sales", connection))

,Field,Type,Null,Key,Default,Extra
0,Transaction_Date,date,YES,,None,
1,Sales_Type,varchar(50),YES,,None,
2,Is_Return,tinyint(1),YES,,None,
3,Reason_of_Return,varchar(200),YES,,None,
4,Supplier,varchar(100),YES,,None,
5,Product_No,varchar(50),YES,,None,
6,Product_Description,text,YES,,None,
7,Product_Division,varchar(150),YES,,None,
8,Product_Category,varchar(150),YES,,None,
9,Product_Subcategory,varchar(150),YES,,None,


In [37]:
create_fact_sales = """
CREATE TABLE IF NOT EXISTS fact_sales (
    sales_key BIGINT AUTO_INCREMENT PRIMARY KEY,
    date_key INT,
    product_key INT,
    store_key INT,
    supplier_key INT,

    sales_type VARCHAR(50),
    is_return TINYINT(1),
    reason_of_return VARCHAR(200),

    qty_sold DECIMAL(12,2),
    sales_amount DECIMAL(14,2),
    cogs DECIMAL(14,2),
    number_of_transactions INT,

    net_profit_contribution DECIMAL(14,2),
    net_quantity DECIMAL(12,2),
    net_sales DECIMAL(14,2),
    net_cogs DECIMAL(14,2),

    flag_negative_quantity TINYINT(1),
    flag_negative_sales TINYINT(1),
    flag_negative_cogs TINYINT(1),

    FOREIGN KEY (date_key) REFERENCES dim_date(date_key),
    FOREIGN KEY (product_key) REFERENCES dim_product(product_key),
    FOREIGN KEY (store_key) REFERENCES dim_store(store_key),
    FOREIGN KEY (supplier_key) REFERENCES dim_supplier(supplier_key)
)
"""

with engine.begin() as connection:
    connection.execute(text(create_fact_sales))

print("fact_sales table created successfully!")

fact_sales table created successfully!


In [38]:
insert_sales = """
INSERT INTO fact_sales (
    date_key,
    product_key,
    store_key,
    supplier_key,
    sales_type,
    is_return,
    reason_of_return,
    qty_sold,
    sales_amount,
    cogs,
    number_of_transactions,
    net_profit_contribution,
    net_quantity,
    net_sales,
    net_cogs,
    flag_negative_quantity,
    flag_negative_sales,
    flag_negative_cogs
)
SELECT
    d.date_key,
    p.product_key,
    st.store_key,
    su.supplier_key,
    s.Sales_Type,
    s.Is_Return,
    s.Reason_of_Return,
    s.Qty_Sold,
    s.Sales_Amount,
    s.Cogs,
    s.Number_of_Transactions,
    s.Net_Profit_Contribution,
    s.Net_Quantity,
    s.Net_Sales,
    s.Net_COGS,
    s.flag_negative_quantity,
    s.flag_negative_sales,
    s.flag_negative_cogs
FROM stg_sales s
LEFT JOIN dim_date d
    ON s.Transaction_Date = d.full_date
LEFT JOIN dim_product p
    ON s.Product_No = p.product_no
LEFT JOIN dim_store st
    ON s.Store = st.store_id
LEFT JOIN dim_supplier su
    ON s.Supplier = su.supplier_id
"""

with engine.begin() as connection:
    connection.execute(text(insert_sales))

print("Sales data inserted successfully!")

Sales data inserted successfully!


In [39]:
sales_audit = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT sales_key) AS unique_sales_rows,
    SUM(is_return = 1) AS total_returns,
    SUM(date_key IS NULL) AS missing_date_keys,
    SUM(product_key IS NULL) AS missing_product_keys,
    SUM(store_key IS NULL) AS missing_store_keys,
    SUM(supplier_key IS NULL) AS missing_supplier_keys,
    ROUND(SUM(net_sales), 2) AS total_net_sales,
    ROUND(SUM(net_profit_contribution), 2) AS total_profit_contribution
FROM fact_sales
"""

with engine.connect() as connection:
    display(pd.read_sql(sales_audit, connection))

,total_rows,unique_sales_rows,total_returns,missing_date_keys,missing_product_keys,missing_store_keys,missing_supplier_keys,total_net_sales,total_profit_contribution
0,125751,125751,7491.0,0.0,0.0,0.0,0.0,10482110.25,4634359.14


In [40]:
with engine.connect() as connection:
    display(pd.read_sql("DESCRIBE stg_demand", connection))

,Field,Type,Null,Key,Default,Extra
0,Date,date,YES,,None,
1,Store_ID,varchar(20),YES,,None,
2,Product_ID,varchar(20),YES,,None,
3,Category,varchar(100),YES,,None,
4,Region,varchar(50),YES,,None,
5,Inventory_Level,int,YES,,None,
6,Units_Sold,int,YES,,None,
7,Units_Ordered,int,YES,,None,
8,Demand_Forecast,"decimal(12,2)",YES,,None,
9,Price,"decimal(12,2)",YES,,None,


In [41]:
create_fact_demand = """
CREATE TABLE IF NOT EXISTS fact_demand (
    demand_key BIGINT AUTO_INCREMENT PRIMARY KEY,

    date_key INT,
    store_id VARCHAR(20),
    product_id VARCHAR(20),

    category VARCHAR(100),
    region VARCHAR(50),

    inventory_level INT,
    units_sold INT,
    units_ordered INT,

    demand_forecast DECIMAL(12,2),
    demand_forecast_clean DECIMAL(12,2),
    flag_negative_forecast TINYINT(1),
    inventory_coverage DECIMAL(12,2),

    price DECIMAL(12,2),
    discount DECIMAL(8,2),
    competitor_pricing DECIMAL(12,2),

    weather_condition VARCHAR(50),
    holiday_promotion TINYINT(1),
    seasonality VARCHAR(50),

    FOREIGN KEY (date_key) REFERENCES dim_date(date_key)
)
"""

with engine.begin() as connection:
    connection.execute(text(create_fact_demand))

print("fact_demand table created successfully!")

fact_demand table created successfully!


In [3]:
insert_demand = """
INSERT INTO fact_demand (
    date_key,
    store_id,
    product_id,
    category,
    region,
    inventory_level,
    units_sold,
    units_ordered,
    demand_forecast,
    demand_forecast_clean,
    flag_negative_forecast,
    inventory_coverage,
    price,
    discount,
    competitor_pricing,
    weather_condition,
    holiday_promotion,
    seasonality
)
SELECT
    d.date_key,
    s.Store_ID,
    s.Product_ID,
    s.Category,
    s.Region,
    s.Inventory_Level,
    s.Units_Sold,
    s.Units_Ordered,
    s.Demand_Forecast,
    s.Demand_Forecast_Clean,
    s.flag_negative_forecast,
    s.Inventory_Coverage,
    s.Price,
    s.Discount,
    s.Competitor_Pricing,
    s.Weather_Condition,
    s.Holiday_Promotion,
    s.Seasonality
FROM stg_demand s
LEFT JOIN dim_date d
    ON s.Date = d.full_date
"""

with engine.begin() as connection:
    connection.execute(text(insert_demand))

print("Demand data inserted successfully!")

Demand data inserted successfully!


In [6]:
demand_audit = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT demand_key) AS unique_rows,
    SUM(date_key IS NULL) AS missing_date_keys,
    SUM(flag_negative_forecast = 1) AS negative_forecast_records,
    SUM(demand_forecast_clean < 0) AS negative_clean_forecasts,
    ROUND(AVG(demand_forecast_clean), 2) AS avg_clean_forecast,
    ROUND(AVG(inventory_coverage), 2) AS avg_inventory_coverage
FROM fact_demand
"""

with engine.connect() as connection:
    display(pd.read_sql(demand_audit, connection))

,total_rows,unique_rows,missing_date_keys,negative_forecast_records,negative_clean_forecasts,avg_clean_forecast,avg_inventory_coverage
0,73100,73100,0.0,673.0,0.0,141.53,6.69


In [7]:
with engine.connect() as connection:
    display(pd.read_sql("DESCRIBE fact_inventory", connection))

,Field,Type,Null,Key,Default,Extra
0,inventory_key,bigint,NO,PRI,None,auto_increment
1,product_key,int,NO,MUL,None,
2,store_key,int,NO,MUL,None,
3,supplier_key,int,NO,MUL,None,
4,start_date_key,int,YES,MUL,None,
5,end_date_key,int,YES,MUL,None,
6,stock_status,varchar(50),YES,,None,
7,quantity_on_hand,int,YES,,None,
8,selling_value,"decimal(14,2)",YES,,None,
9,cost_value,"decimal(14,2)",YES,,None,


In [8]:
inventory_kpi_query = """
SELECT
    COUNT(*) AS total_inventory_records,

    COUNT(DISTINCT product_key) AS unique_products,
    COUNT(DISTINCT store_key) AS unique_stores,
    COUNT(DISTINCT supplier_key) AS unique_suppliers,

    SUM(quantity_on_hand < 0) AS negative_stock_records,
    SUM(quantity_on_hand = 0) AS zero_stock_records,

    ROUND(AVG(quantity_on_hand), 2) AS avg_quantity_on_hand,

    ROUND(SUM(selling_value), 2) AS total_recorded_selling_value,
    ROUND(SUM(cost_value), 2) AS total_recorded_cost_value,

    COUNT(DISTINCT start_date_key) AS distinct_start_dates
FROM fact_inventory
"""

with engine.connect() as connection:
    display(pd.read_sql(inventory_kpi_query, connection))

,total_inventory_records,unique_products,unique_stores,unique_suppliers,negative_stock_records,zero_stock_records,avg_quantity_on_hand,total_recorded_selling_value,total_recorded_cost_value,distinct_start_dates
0,284755,2326,40,30,1380.0,13869.0,2.29,63896905.11,31620976.61,328


In [9]:
stock_status_query = """
SELECT
    stock_status,
    COUNT(*) AS total_records,
    SUM(quantity_on_hand) AS total_quantity,
    ROUND(AVG(quantity_on_hand), 2) AS avg_quantity,
    ROUND(SUM(selling_value), 2) AS recorded_selling_value,
    ROUND(SUM(cost_value), 2) AS recorded_cost_value
FROM fact_inventory
GROUP BY stock_status
ORDER BY total_records DESC
"""

with engine.connect() as connection:
    display(pd.read_sql(stock_status_query, connection))

,stock_status,total_records,total_quantity,avg_quantity,recorded_selling_value,recorded_cost_value
0,Full Price,261899,603868.0,2.31,62421120.00,30175598.22
1,Markdown Tier 1,12091,23900.0,1.98,920646.00,795223.39
2,Markdown Tier 2,6937,15440.0,2.23,439710.60,453536.05
3,Clearance,3798,10149.0,2.67,113178.51,195583.95
4,Promo,30,50.0,1.67,2250.00,1035.00


In [10]:
margin_analysis_query = """
SELECT
    stock_status,
    COUNT(*) AS total_records,

    ROUND(SUM(selling_value), 2) AS selling_value,
    ROUND(SUM(cost_value), 2) AS cost_value,

    ROUND(SUM(selling_value - cost_value), 2)
        AS recorded_value_difference,

    ROUND(
        100 * SUM(selling_value - cost_value)
        / NULLIF(SUM(selling_value), 0), 2
    ) AS recorded_margin_percentage,

    SUM(quantity_on_hand < 0) AS negative_stock_records,
    SUM(quantity_on_hand = 0) AS zero_stock_records

FROM fact_inventory
GROUP BY stock_status
ORDER BY recorded_value_difference ASC
"""

with engine.connect() as connection:
    display(pd.read_sql(margin_analysis_query, connection))

,stock_status,total_records,selling_value,cost_value,recorded_value_difference,recorded_margin_percentage,negative_stock_records,zero_stock_records
0,Clearance,3798,113178.51,195583.95,-82405.44,-72.81,60.0,51.0
1,Markdown Tier 2,6937,439710.60,453536.05,-13825.45,-3.14,81.0,593.0
2,Promo,30,2250.00,1035.00,1215.00,54.00,0.0,0.0
3,Markdown Tier 1,12091,920646.00,795223.39,125422.61,13.62,161.0,572.0
4,Full Price,261899,62421120.00,30175598.22,32245521.78,51.66,1078.0,12653.0


In [11]:
stock_anomaly_query = """
SELECT
    stock_status,

    COUNT(*) AS total_records,

    SUM(CASE
        WHEN quantity_on_hand < 0 THEN 1 ELSE 0
    END) AS negative_stock_records,

    SUM(CASE
        WHEN quantity_on_hand = 0 THEN 1 ELSE 0
    END) AS zero_stock_records,

    SUM(CASE
        WHEN unit_cost_price < 0 THEN 1 ELSE 0
    END) AS negative_unit_cost_records,

    SUM(CASE
        WHEN unit_selling_price = 0 THEN 1 ELSE 0
    END) AS zero_selling_price_records

FROM fact_inventory
GROUP BY stock_status
ORDER BY negative_stock_records DESC
"""

with engine.connect() as connection:
    display(pd.read_sql(stock_anomaly_query, connection))

,stock_status,total_records,negative_stock_records,zero_stock_records,negative_unit_cost_records,zero_selling_price_records
0,Full Price,261899,1078.0,12653.0,23.0,12653.0
1,Markdown Tier 1,12091,161.0,572.0,2.0,572.0
2,Markdown Tier 2,6937,81.0,593.0,0.0,593.0
3,Clearance,3798,60.0,51.0,0.0,51.0
4,Promo,30,0.0,0.0,0.0,0.0


In [12]:
store_inventory_query = """
SELECT
    st.store_id,

    COUNT(*) AS inventory_records,

    COUNT(DISTINCT fi.product_key) AS unique_products,

    SUM(CASE
        WHEN fi.quantity_on_hand < 0 THEN 1 ELSE 0
    END) AS negative_stock_records,

    SUM(CASE
        WHEN fi.quantity_on_hand = 0 THEN 1 ELSE 0
    END) AS zero_stock_records,

    ROUND(SUM(fi.selling_value), 2)
        AS recorded_selling_value,

    ROUND(SUM(fi.cost_value), 2)
        AS recorded_cost_value,

    ROUND(AVG(fi.quantity_on_hand), 2)
        AS average_quantity

FROM fact_inventory fi
JOIN dim_store st
    ON fi.store_key = st.store_key

GROUP BY st.store_id
ORDER BY recorded_selling_value DESC
"""

with engine.connect() as connection:
    display(pd.read_sql(store_inventory_query, connection))

,store_id,inventory_records,unique_products,negative_stock_records,zero_stock_records,recorded_selling_value,recorded_cost_value,average_quantity
0,STR-1006,11142,1978,48.0,633.0,2729701.91,1319815.28,2.58
1,STR-1229,7913,1401,22.0,512.0,2199976.69,1087974.83,2.44
2,STR-1010,9992,2010,53.0,431.0,2084988.09,1033462.30,2.44
3,STR-1044,9241,1931,69.0,403.0,2044730.31,998888.99,2.37
4,STR-1198,11582,2007,96.0,449.0,2031560.92,1045904.99,2.71
5,STR-1222,8269,1819,69.0,403.0,1999878.11,981585.27,2.27
6,STR-1094,8663,1743,41.0,383.0,1981819.72,982142.55,2.40
7,STR-1109,8908,1893,35.0,301.0,1944595.30,959684.06,2.40
8,STR-1148,8796,1843,41.0,468.0,1890075.20,932555.82,2.41
9,STR-1080,7037,1324,30.0,499.0,1856854.20,916544.92,2.34


In [13]:
store_risk_query = """
SELECT
    st.store_id,
    COUNT(*) AS total_records,

    SUM(CASE
        WHEN fi.quantity_on_hand < 0 THEN 1 ELSE 0
    END) AS negative_stock_records,

    ROUND(
        100.0 * SUM(CASE
            WHEN fi.quantity_on_hand < 0 THEN 1 ELSE 0
        END) / COUNT(*), 2
    ) AS negative_stock_rate,

    SUM(CASE
        WHEN fi.quantity_on_hand = 0 THEN 1 ELSE 0
    END) AS zero_stock_records,

    ROUND(
        100.0 * SUM(CASE
            WHEN fi.quantity_on_hand = 0 THEN 1 ELSE 0
        END) / COUNT(*), 2
    ) AS zero_stock_rate

FROM fact_inventory fi
JOIN dim_store st
    ON fi.store_key = st.store_key

GROUP BY st.store_id
ORDER BY negative_stock_rate DESC
"""

with engine.connect() as connection:
    display(pd.read_sql(store_risk_query, connection))

,store_id,total_records,negative_stock_records,negative_stock_rate,zero_stock_records,zero_stock_rate
0,STR-1157,7692,81.0,1.05,338.0,4.39
1,STR-1255,7051,72.0,1.02,232.0,3.29
2,STR-1085,5861,60.0,1.02,323.0,5.51
3,STR-1163,5885,52.0,0.88,343.0,5.83
4,STR-1222,8269,69.0,0.83,403.0,4.87
5,STR-1198,11582,96.0,0.83,449.0,3.88
6,STR-1044,9241,69.0,0.75,403.0,4.36
7,STR-1253,7711,43.0,0.56,282.0,3.66
8,STR-1235,6570,36.0,0.55,392.0,5.97
9,STR-1194,8594,46.0,0.54,450.0,5.24


In [14]:
abc_analysis_query = """
WITH product_sales AS (
    SELECT
        p.product_key,
        p.product_no,
        p.product_description,
        SUM(s.net_sales) AS total_net_sales
    FROM fact_sales s
    JOIN dim_product p
        ON s.product_key = p.product_key
    GROUP BY
        p.product_key,
        p.product_no,
        p.product_description
),

ranked_products AS (
    SELECT
        *,
        SUM(total_net_sales) OVER () AS overall_sales,

        SUM(total_net_sales) OVER (
            ORDER BY total_net_sales DESC, product_key
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_sales

    FROM product_sales
)

SELECT
    product_no,
    product_description,
    ROUND(total_net_sales, 2) AS total_net_sales,

    ROUND(
        100.0 * total_net_sales / NULLIF(overall_sales, 0), 2
    ) AS sales_contribution_pct,

    ROUND(
        100.0 * cumulative_sales / NULLIF(overall_sales, 0), 2
    ) AS cumulative_sales_pct,

    CASE
        WHEN cumulative_sales / NULLIF(overall_sales, 0) <= 0.80
            THEN 'A'
        WHEN cumulative_sales / NULLIF(overall_sales, 0) <= 0.95
            THEN 'B'
        ELSE 'C'
    END AS abc_category

FROM ranked_products
ORDER BY total_net_sales DESC
"""

with engine.connect() as connection:
    abc_products = pd.read_sql(abc_analysis_query, connection)

display(abc_products.head(20))

,product_no,product_description,total_net_sales,sales_contribution_pct,cumulative_sales_pct,abc_category
0,PROD-140871,PROD-100439 SEG-500064 Scholar Footwear Boot C...,104027.50,0.99,0.99,A
1,PROD-100805,PROD-102578 SEG-500351 Scholar Footwear Flight...,102481.99,0.98,1.97,A
2,PROD-100804,PROD-102576 SEG-500351 Scholar Footwear Flight...,72822.50,0.69,2.66,A
3,PROD-100803,PROD-102577 SEG-500351 Scholar Footwear Flight...,71496.48,0.68,3.35,A
4,PROD-140870,PROD-100438 SEG-500064 Scholar Footwear Boot C...,61705.00,0.59,3.94,A
5,PROD-175338,PROD-113366 SEG-502458 Heritage Footwear Retro...,60040.00,0.57,4.51,A
6,PROD-100802,PROD-102574 SEG-500351 Scholar Footwear Flight...,56361.50,0.54,5.05,A
7,PROD-140869,PROD-100437 SEG-500064 Scholar Footwear Boot C...,55251.90,0.53,5.57,A
8,PROD-175334,PROD-113357 SEG-502458 Heritage Footwear Retro...,51110.00,0.49,6.06,A
9,PROD-152143,PROD-162923 SEG-514991 Heritage Footwear Retro...,45672.00,0.44,6.50,A


In [17]:
display(abc_products)

,product_no,product_description,total_net_sales,sales_contribution_pct,cumulative_sales_pct,abc_category
0,PROD-140871,PROD-100439 SEG-500064 Scholar Footwear Boot C...,104027.50,0.99,0.99,A
1,PROD-100805,PROD-102578 SEG-500351 Scholar Footwear Flight...,102481.99,0.98,1.97,A
2,PROD-100804,PROD-102576 SEG-500351 Scholar Footwear Flight...,72822.50,0.69,2.66,A
3,PROD-100803,PROD-102577 SEG-500351 Scholar Footwear Flight...,71496.48,0.68,3.35,A
4,PROD-140870,PROD-100438 SEG-500064 Scholar Footwear Boot C...,61705.00,0.59,3.94,A
...,...,...,...,...,...,...
2321,PROD-148307,PROD-117613 SEG-503080 Junior Apparel Printed ...,278.51,0.00,99.99,C
2322,PROD-168841,PROD-170350 SEG-516327 Junior Apparel Crew Tee...,277.89,0.00,99.99,C
2323,PROD-151682,PROD-170224 SEG-516298 Junior Apparel Cozy Tro...,277.44,0.00,99.99,C
2324,PROD-190340,PROD-137217 SEG-508422 Junior Apparel Crew Tee...,275.80,0.00,100.00,C


In [18]:
abc_summary = (
    abc_products.groupby("abc_category")
    .agg(
        product_count=("product_no", "count"),
        total_net_sales=("total_net_sales", "sum")
    )
    .reset_index()
)

abc_summary["product_share_pct"] = (
    abc_summary["product_count"]
    / len(abc_products) * 100
).round(2)

abc_summary["sales_share_pct"] = (
    abc_summary["total_net_sales"]
    / abc_products["total_net_sales"].sum() * 100
).round(2)

display(abc_summary)

,abc_category,product_count,total_net_sales,product_share_pct,sales_share_pct
0,A,928,8384530.49,39.90,79.99
1,B,601,1572385.46,25.84,15.00
2,C,797,525194.30,34.26,5.01


In [19]:
abc_inventory_risk_query = """
WITH product_sales AS (
    SELECT
        p.product_key,
        p.product_no,
        p.product_description,
        SUM(s.net_sales) AS total_net_sales
    FROM fact_sales s
    JOIN dim_product p
        ON s.product_key = p.product_key
    GROUP BY
        p.product_key,
        p.product_no,
        p.product_description
),

ranked_products AS (
    SELECT
        *,
        SUM(total_net_sales) OVER (
            ORDER BY total_net_sales DESC, product_key
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_sales,
        SUM(total_net_sales) OVER () AS overall_sales
    FROM product_sales
),

abc_products AS (
    SELECT
        *,
        CASE
            WHEN cumulative_sales / NULLIF(overall_sales, 0) <= 0.80
                THEN 'A'
            WHEN cumulative_sales / NULLIF(overall_sales, 0) <= 0.95
                THEN 'B'
            ELSE 'C'
        END AS abc_category
    FROM ranked_products
),

inventory_risk AS (
    SELECT
        product_key,
        COUNT(*) AS inventory_records,

        SUM(CASE
            WHEN quantity_on_hand < 0 THEN 1 ELSE 0
        END) AS negative_stock_records,

        SUM(CASE
            WHEN quantity_on_hand = 0 THEN 1 ELSE 0
        END) AS zero_stock_records

    FROM fact_inventory
    GROUP BY product_key
)

SELECT
    a.product_no,
    a.abc_category,
    ROUND(a.total_net_sales, 2) AS total_net_sales,

    COALESCE(i.inventory_records, 0) AS inventory_records,
    COALESCE(i.negative_stock_records, 0) AS negative_stock_records,
    COALESCE(i.zero_stock_records, 0) AS zero_stock_records

FROM abc_products a
LEFT JOIN inventory_risk i
    ON a.product_key = i.product_key

WHERE a.abc_category = 'A'

ORDER BY
    COALESCE(i.zero_stock_records, 0) DESC,
    COALESCE(i.negative_stock_records, 0) DESC

LIMIT 20
"""

with engine.connect() as connection:
    display(pd.read_sql(abc_inventory_risk_query, connection))

,product_no,abc_category,total_net_sales,inventory_records,negative_stock_records,zero_stock_records
0,PROD-166368,A,34923.00,715,2.0,107.0
1,PROD-166365,A,27434.75,677,2.0,98.0
2,PROD-166367,A,22568.00,592,6.0,95.0
3,PROD-166366,A,39218.75,801,1.0,90.0
4,PROD-166364,A,15390.50,497,5.0,87.0
5,PROD-166363,A,14273.50,470,10.0,86.0
6,PROD-166362,A,15927.75,487,0.0,85.0
7,PROD-100359,A,9987.70,378,2.0,84.0
8,PROD-121532,A,10712.00,356,1.0,82.0
9,PROD-100433,A,16340.06,667,0.0,79.0


In [20]:
top_product_store_query = """
SELECT
    p.product_no,
    st.store_id,

    COUNT(*) AS inventory_records,

    SUM(CASE
        WHEN fi.quantity_on_hand = 0 THEN 1 ELSE 0
    END) AS zero_stock_records,

    SUM(CASE
        WHEN fi.quantity_on_hand < 0 THEN 1 ELSE 0
    END) AS negative_stock_records,

    ROUND(
        100.0 * SUM(CASE
            WHEN fi.quantity_on_hand = 0 THEN 1 ELSE 0
        END) / COUNT(*), 2
    ) AS zero_stock_rate

FROM fact_inventory fi

JOIN dim_product p
    ON fi.product_key = p.product_key

JOIN dim_store st
    ON fi.store_key = st.store_key

WHERE p.product_no IN (
    'PROD-166368',
    'PROD-166365',
    'PROD-166367'
)

GROUP BY
    p.product_no,
    st.store_id

HAVING COUNT(*) > 0

ORDER BY
    p.product_no,
    zero_stock_records DESC
"""

with engine.connect() as connection:
    display(pd.read_sql(top_product_store_query, connection))

,product_no,store_id,inventory_records,zero_stock_records,negative_stock_records,zero_stock_rate
0,PROD-166365,STR-1198,20,5.0,0.0,25.00
1,PROD-166365,STR-1075,29,5.0,0.0,17.24
2,PROD-166365,STR-1335,17,5.0,0.0,29.41
3,PROD-166365,STR-1148,39,5.0,0.0,12.82
4,PROD-166365,STR-1107,22,4.0,2.0,18.18
...,...,...,...,...,...,...
115,PROD-166368,STR-1105,15,1.0,0.0,6.67
116,PROD-166368,STR-1074,12,1.0,0.0,8.33
117,PROD-166368,STR-1145,15,1.0,0.0,6.67
118,PROD-166368,STR-1082,18,0.0,0.0,0.00


In [22]:
risk_ranking = pd.read_sql(
    top_product_store_query,
    engine
)

risk_ranking = risk_ranking[
    risk_ranking["inventory_records"] >= 10
].sort_values(
    ["zero_stock_rate", "zero_stock_records"],
    ascending=False
)

display(risk_ranking.head(10))



product_risk_summary = (
    risk_ranking.groupby("product_no")
    .agg(
        stores_analysed=("store_id", "nunique"),
        total_observations=("inventory_records", "sum"),
        zero_stock_records=("zero_stock_records", "sum"),
        avg_zero_stock_rate=("zero_stock_rate", "mean"),
        highest_zero_stock_rate=("zero_stock_rate", "max")
    )
    .reset_index()
    .sort_values("avg_zero_stock_rate", ascending=False)
)

display(product_risk_summary)

,product_no,store_id,inventory_records,zero_stock_records,negative_stock_records,zero_stock_rate
46,PROD-166367,STR-1044,12,5.0,0.0,41.67
42,PROD-166367,STR-1203,14,5.0,0.0,35.71
43,PROD-166367,STR-1107,15,5.0,0.0,33.33
5,PROD-166365,STR-1044,12,4.0,0.0,33.33
81,PROD-166368,STR-1149,19,6.0,0.0,31.58
44,PROD-166367,STR-1114,16,5.0,0.0,31.25
45,PROD-166367,STR-1157,16,5.0,1.0,31.25
87,PROD-166368,STR-1044,13,4.0,0.0,30.77
2,PROD-166365,STR-1335,17,5.0,0.0,29.41
84,PROD-166368,STR-1203,17,5.0,0.0,29.41


,product_no,stores_analysed,total_observations,zero_stock_records,avg_zero_stock_rate,highest_zero_stock_rate
1,PROD-166367,34,539,90.0,16.239118,41.67
2,PROD-166368,39,708,107.0,15.246154,31.58
0,PROD-166365,40,677,98.0,14.766000,33.33


In [23]:
supplier_risk_query = """
SELECT
    p.product_no,
    su.supplier_id,
    su.supplier_name,

    COUNT(*) AS inventory_records,

    SUM(CASE
        WHEN fi.quantity_on_hand = 0 THEN 1 ELSE 0
    END) AS zero_stock_records,

    SUM(CASE
        WHEN fi.quantity_on_hand < 0 THEN 1 ELSE 0
    END) AS negative_stock_records,

    ROUND(
        100.0 * SUM(CASE
            WHEN fi.quantity_on_hand = 0 THEN 1 ELSE 0
        END) / COUNT(*), 2
    ) AS zero_stock_rate

FROM fact_inventory fi

JOIN dim_product p
    ON fi.product_key = p.product_key

JOIN dim_supplier su
    ON fi.supplier_key = su.supplier_key

WHERE p.product_no IN (
    'PROD-166367',
    'PROD-166368',
    'PROD-166365'
)

GROUP BY
    p.product_no,
    su.supplier_id,
    su.supplier_name

ORDER BY
    zero_stock_rate DESC
"""

with engine.connect() as connection:
    display(pd.read_sql(supplier_risk_query, connection))

,product_no,supplier_id,supplier_name,inventory_records,zero_stock_records,negative_stock_records,zero_stock_rate
0,PROD-166367,Vendor 0169,Vendor 0169,592,95.0,6.0,16.05
1,PROD-166368,Vendor 0169,Vendor 0169,715,107.0,2.0,14.97
2,PROD-166365,Vendor 0169,Vendor 0169,677,98.0,2.0,14.48


In [26]:
vendor_portfolio_query = """
SELECT
    p.product_no,
    COUNT(*) AS inventory_records,

    SUM(CASE
        WHEN fi.quantity_on_hand = 0 THEN 1 ELSE 0
    END) AS zero_stock_records,

    SUM(CASE
        WHEN fi.quantity_on_hand < 0 THEN 1 ELSE 0
    END) AS negative_stock_records,

    ROUND(
        100.0 * SUM(CASE
            WHEN fi.quantity_on_hand = 0 THEN 1 ELSE 0
        END) / COUNT(*), 2
    ) AS zero_stock_rate

FROM fact_inventory fi

JOIN dim_product p
    ON fi.product_key = p.product_key

JOIN dim_supplier su
    ON fi.supplier_key = su.supplier_key

WHERE su.supplier_id = 'Vendor 0169'

GROUP BY p.product_no

ORDER BY zero_stock_rate DESC
LIMIT 20
"""

with engine.connect() as connection:
    display(pd.read_sql(vendor_portfolio_query, connection))

,product_no,inventory_records,zero_stock_records,negative_stock_records,zero_stock_rate
0,PROD-156685,53,24.0,2.0,45.28
1,PROD-156686,55,22.0,1.0,40.00
2,PROD-152239,68,24.0,0.0,35.29
3,PROD-156694,66,23.0,0.0,34.85
4,PROD-152240,78,26.0,1.0,33.33
5,PROD-152237,74,24.0,0.0,32.43
6,PROD-165960,81,26.0,0.0,32.10
7,PROD-179164,13,4.0,0.0,30.77
8,PROD-165959,72,22.0,0.0,30.56
9,PROD-156693,56,17.0,0.0,30.36


In [27]:
supplier_comparison_query = """
SELECT
    su.supplier_id,
    su.supplier_name,

    COUNT(*) AS inventory_records,

    COUNT(DISTINCT fi.product_key) AS products_supplied,

    SUM(CASE
        WHEN fi.quantity_on_hand = 0 THEN 1 ELSE 0
    END) AS zero_stock_records,

    SUM(CASE
        WHEN fi.quantity_on_hand < 0 THEN 1 ELSE 0
    END) AS negative_stock_records,

    ROUND(
        100.0 * SUM(CASE
            WHEN fi.quantity_on_hand = 0 THEN 1 ELSE 0
        END) / COUNT(*), 2
    ) AS zero_stock_rate

FROM fact_inventory fi

JOIN dim_supplier su
    ON fi.supplier_key = su.supplier_key

GROUP BY
    su.supplier_id,
    su.supplier_name

HAVING COUNT(*) >= 500

ORDER BY zero_stock_rate DESC
"""

with engine.connect() as connection:
    supplier_comparison = pd.read_sql(
        supplier_comparison_query,
        connection
    )

display(supplier_comparison)

,supplier_id,supplier_name,inventory_records,products_supplied,zero_stock_records,negative_stock_records,zero_stock_rate
0,Vendor 0262,Vendor 0262,3673,48,353.0,30.0,9.61
1,Vendor 0010,Vendor 0010,19351,134,1501.0,98.0,7.76
2,Vendor 0169,Vendor 0169,42390,309,3231.0,121.0,7.62
3,Vendor 0008,Vendor 0008,1070,21,81.0,7.0,7.57
4,Vendor 0166,Vendor 0166,100747,716,5229.0,350.0,5.19
5,Vendor 0134,Vendor 0134,49764,391,2564.0,201.0,5.15
6,Vendor 0064,Vendor 0064,10104,85,413.0,32.0,4.09
7,Vendor 0054,Vendor 0054,3178,10,82.0,113.0,2.58
8,Vendor 0258,Vendor 0258,8556,45,199.0,13.0,2.33
9,Vendor 0155,Vendor 0155,551,7,9.0,11.0,1.63


In [28]:
overall_supplier_rate = (
    supplier_comparison["zero_stock_records"].sum()
    / supplier_comparison["inventory_records"].sum()
    * 100
)

vendor_0169_rate = supplier_comparison.loc[
    supplier_comparison["supplier_id"] == "Vendor 0169",
    "zero_stock_rate"
].iloc[0]

print(f"Overall supplier zero-stock rate: {overall_supplier_rate:.2f}%")
print(f"Vendor 0169 zero-stock rate: {vendor_0169_rate:.2f}%")
print(f"Difference: {vendor_0169_rate - overall_supplier_rate:+.2f} percentage points")

Overall supplier zero-stock rate: 4.89%
Vendor 0169 zero-stock rate: 7.62%
Difference: +2.73 percentage points
